# 한국무브넥스 ECT — 처음부터 분석하기

**처음에는 `Run → Run All Cells`를 눌러 연습용 실행을 확인하세요.** 한 칸씩 보려면 칸을 클릭한 뒤 `Shift + Enter`입니다.

진행 순서: **① 준비 → ② 데이터 선택 → ③ 수량 확인 → ④ 날짜 구분 → ⑤ 기존 기준 → ⑥ ML 비교 → ⑦ 그래프 → ⑧ 저장**.

실제 회사 데이터는 별도 전달 파일을 `data/private`에 넣고 아래 `DATA_MODE`를 `"company"`로 바꿉니다. PC 기본값 `"demo"`는 합성 데이터입니다. Docker에서는 서버의 회사 CSV 설정을 사용합니다.

이 노트북은 위치 1·2를 따로 계산합니다. 결과는 **가성 후보 / 보류 / 진성 의심**이며 최종 처분은 현장 확인이 필요합니다.

## ① 실행 준비
이 칸을 먼저 실행합니다. 프로그램을 불러오는 준비 단계입니다.

In [ ]:
from pathlib import Path
import sys, os
ROOT = Path(os.environ.get("ECT_PROJECT_ROOT", str(Path.cwd())))
if not (ROOT / "ectlab").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "ectlab").is_dir():
    raise RuntimeError("프로젝트의 notebooks 폴더에서 열어 주세요.")
sys.path.insert(0, str(ROOT))
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
from IPython.display import display, HTML, FileLink
import pandas as pd
from ectlab.runtime import data_path
from ectlab.data import load_csv, inventory
from ectlab.pipeline import Settings, prepare, distance_analysis, train_models, summarize, export_run
from ectlab.plots import pca_plot, comparison_plot, true_case_plot, frequency_plot
print("실행 준비 완료. 다음 칸을 실행하세요.")

## ② 사용할 데이터 선택
Docker에서는 서버 설정을 자동으로 읽습니다. PC에서는 처음에 연습용으로 실행하고, 회사 데이터가 준비되면 `DATA_MODE = "company"`로 바꾸고 파일명을 확인하세요.

In [ ]:
DATA_MODE = os.environ.get("ECT_DATA_MODE", "demo")  # "demo": 연습용 / "company": 실제 회사 데이터
PRODUCT = os.environ.get("ECT_PRODUCT", "C10A_49513-75270R")
DATA_PATH = data_path(ROOT, DATA_MODE, PRODUCT)
if DATA_MODE not in ("demo", "company"):
    raise ValueError('DATA_MODE는 "demo" 또는 "company"입니다.')
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"{DATA_PATH.name}이 없습니다. 별도 전달받은 CSV를 data/private에 넣어 주세요.")
df = load_csv(DATA_PATH)
message = "연습용 합성 데이터입니다. 실제 회사 성능으로 사용하지 마세요." if DATA_MODE == "demo" else "실제 회사 데이터입니다. 공유 전 회사 데이터가 포함되는지 확인하세요."
display(HTML(f'<div style="padding:16px;border:2px solid #ba7900;background:#fff4d6;font-size:19px"><b>{message}</b></div>'))
print("선택한 파일:", DATA_PATH.name)

## ③ 데이터 수량과 품질 확인
‘측정행’은 위치 한 곳에서 측정한 한 줄입니다. 제품 개수와 같다고 가정하지 않습니다. 0/노랑, 측정 오류, 라벨 미확인 행은 학습 대상에서 제외됩니다.

In [ ]:
display(inventory(df))
display(df.groupby(["date", "position", "field_label"]).size().rename("측정행 수").reset_index())

## ④ 학습 날짜 / 기준 설정 날짜 / 시험 날짜 구분

- 학습: 과거 데이터로 모델과 정상 범위를 만듭니다.
- 기준 설정: 다음 시점의 데이터로 거리 기준을 정합니다.
- 시험: 그 이후의 날짜로 결과를 비교합니다.

기존 결과 재현 시 아래 날짜를 그대로 사용하세요. 추가 데이터로 **재학습할 때만** 날짜를 의도적으로 바꾸고, 학습 이후의 시험 날짜를 남겨 둡니다.

In [ ]:
settings = Settings(
    train_end="2026-08-20",
    calibration_end="2026-10-07",
    seed=42,
    c10a_retrospective_guard=True,
)
packs, counts = prepare(df, settings)
display(counts)
display(df.assign(구간=df.date.map(lambda d: "학습" if d <= settings.train_end else "기준 설정" if d <= settings.calibration_end else "시험")).groupby(["구간", "date", "position"]).size().rename("측정행 수").reset_index())

## ⑤ 기존 거리 기반 기준 실행
정상 OK와 마스터에서 얼마나 떨어져 있는지 네 가지 방법으로 비교합니다. 네 방법 중 세 방법 이상이 동의하는지 봅니다.

C10A 위치 2의 첫 블록 30Hz 보완 기준은 **기존 진성 2행을 확인한 뒤 만든 기준**입니다. 이 결과는 사후 확인이며 새 날짜의 진성 데이터로 검증해야 합니다.

In [ ]:
distance_result = distance_analysis(packs)
for message in distance_result["notes"]:
    print(message)
display(summarize(distance_result["predictions"]))

## ⑥ Random Forest / XGBoost 실행
**마스터 기반 비교(`master_proxy`)**: 마스터를 ‘마스터 쪽’, 설비 OK와 학습 날짜의 기업 가성을 ‘반대쪽’으로 학습합니다. 기업 진성 행과 같은 사건그룹은 학습에서 제외합니다.

**기업 라벨 직접 학습(`company_direct`)**: 기업이 구분한 진성/가성으로 학습합니다. 위치 1에는 진성이 없어 생략합니다. 위치 2의 진성 2행은 학습에 포함되므로 이 두 행의 결과를 독립 검증으로 해석하면 안 됩니다.

입력은 해당 위치의 주파수별 **X/Y 값만**입니다. 색상·날짜·카운터·최대값은 입력으로 쓰지 않습니다.

In [ ]:
ml_result = train_models(packs)
for message in ml_result["notes"]:
    print(message)
all_predictions = pd.concat([distance_result["predictions"], ml_result["predictions"]], ignore_index=True)
summary = summarize(all_predictions)
display(summary[summary["비교 구간"].eq("시험 날짜 NG")])

## ⑦ 그래프로 비교
PCA는 여러 입력을 두 축에 펼쳐 보여 주는 그림입니다. 분류 모델과 다릅니다. 전체 NG 그래프에는 학습 행이 포함됩니다. 시험 날짜 그래프를 함께 보세요.

In [ ]:
figures = {}
figures["01_PCA_위치별_분포"] = pca_plot(packs)
display(figures["01_PCA_위치별_분포"])
figures["02_전체NG_가성구분"] = comparison_plot(all_predictions)
display(figures["02_전체NG_가성구분"])
figures["03_시험날짜_가성구분"] = comparison_plot(all_predictions, temporal=True)
display(figures["03_시험날짜_가성구분"])
figures["04_진성행_점수"] = true_case_plot(all_predictions)
display(figures["04_진성행_점수"])
import matplotlib.pyplot as plt
plt.close("all")

## ⑦-2 위치별·주파수별 원래 값 확인
아래 세 값을 바꾸면 해당 위치의 X/Y 그래프를 볼 수 있습니다. 블록은 원본 8슬롯 단위 묶음이며 실제 설비 채널 대응은 추가 확인 대상입니다.

In [ ]:
POSITION = 2
BLOCK = 1
FREQUENCY = "30.0Hz"
figures["05_주파수_XY_비교"] = frequency_plot(packs, POSITION, BLOCK, FREQUENCY)
display(figures["05_주파수_XY_비교"])
plt.close("all")

## ⑧ 결과 저장 — 이 칸까지 실행하면 완료입니다
`results/실행시각` 폴더가 새로 만들어집니다. 이전 결과는 덮어쓰지 않습니다.

**바로 볼 파일**: `결과보고서.html` / **엑셀에서 볼 파일**: `01_위치별_결과요약.csv`, `02_행별_판정근거.csv` / **발표용 그림**: PNG 파일.

결과 CSV에는 학습에 사용한 행인지, 시험 날짜 행인지가 표시됩니다. 저장된 모델은 02번 노트북에서 새 데이터에 적용할 수 있습니다.

In [ ]:
saved_folder = export_run(ROOT / "results", DATA_PATH, settings, packs, distance_result, ml_result, figures)
print("프로젝트 왼쪽 파일 목록에서 results →", saved_folder.name, "폴더를 여세요.")
print("보고서:", saved_folder / "결과보고서.html")
display(summary[summary["비교 구간"].eq("전체 NG: 학습 포함")])

## 결과를 읽을 때 기억할 세 가지
1. 가성 후보는 현장 재검토 대상입니다. 실제 양품을 확정하는 결과가 아닙니다.
2. 현재 C10A 진성은 위치 2의 2행이며 동일 날짜·카운터입니다. 독립적인 진성 제품 수가 충분하지 않습니다.
3. 시험 날짜에는 기업 진성이 없어 진성 검출률을 확인할 수 없습니다. 다른 날짜의 진성과 최종 처분 결과가 추가로 필요합니다.